# **MASTER BUDDY - AI**

## StudyBuddy-AI (DPO Preference-Aligned Study Assistant)

## Overview

StudyBuddy-AI is an AI-powered educational assistant designed to help students understand complex topics in a **clear, structured, and beginner-friendly way**. Unlike standard chatbots, this project focuses on **preference-aligned learning using Direct Preference Optimization (DPO)** to ensure the model consistently produces high-quality explanations.

The system is trained to prefer answers that are:
- Simple and easy to understand
- Well-structured with step-by-step reasoning
- Supported with examples where possible
- Free from unnecessary technical jargon

---

## Objective

The main goal of this project is to build a **study assistant that learns from human preferences**, specifically teaching the model:

> “Good explanations should feel like a patient tutor, not a technical manual.”

---

## Core Concept

This project uses **Direct Preference Optimization (DPO)** to fine-tune a language model based on comparison pairs:

For each question:
- **Prompt** → Student question
- **Chosen response** → Clear, structured, student-friendly explanation
- **Rejected response** → Confusing, overly technical, or poorly structured answer

This allows the model to learn *what makes a good educational explanation*.

---

## Example Training Pair

**Prompt:**
> Explain TCP vs UDP

**Chosen Answer:**
- Simple comparison
- Clear structure
- Real-world examples

**Rejected Answer:**
- Dense technical jargon
- No formatting
- Hard to understand

---

## Tech Stack

- 🧠 Hugging Face Transformers  
- ⚙️ TRL (DPOTrainer)  
- 🤖 LLaMA / Mistral small models  
- 📊 Custom preference dataset  
- 🐍 Python (PyTorch backend)

---

## Key Features

- Preference-aligned fine-tuning using DPO
- Educational chatbot behavior optimization
- Dataset creation for instruction + preference pairs
- Lightweight and scalable training pipeline
- Focus on real-world student learning experience

---

## Expected Outcome

After training, StudyBuddy-AI will:
- Generate clearer explanations than base models
- Prefer structured teaching-style responses
- Act like a **personal AI tutor**

---

## Future Improvements

- Add multi-language support
- Integrate RAG for textbook knowledge
- Expand dataset with real student feedback
- Deploy as web app (React + FastAPI)

---

## Why This Project Matters

This project demonstrates how **AI alignment techniques like DPO** can directly improve real-world usability, especially in education where clarity is more important than complexity.

#### Import Libraries

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer
from datasets import load_dataset
import json
import dotenv
from huggingface_hub import login

##### Helper Function

In [2]:
HF_Token = dotenv.get_key("../.env", "HF_TOKEN")

In [3]:
login(HF_Token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
def get_dataset(filepath):
    data = []
    
    with open(filepath, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                data.append(line)
    return data

In [5]:
dataset = get_dataset("../dataset/dpo_study_assistant_500.jsonl")
dataset[0:10]

['{"prompt": "Explain TCP vs UDP", "chosen": "TCP is connection-oriented and reliable, UDP is faster but unreliable and connectionless. This is important for exams and practical understanding.", "rejected": "TCP and UDP are networking protocols."}',
 '{"prompt": "Give a simple explanation of database normalization", "chosen": "Normalization is organizing database tables to reduce redundancy and improve integrity. This is important for exams and practical understanding.", "rejected": "Normalization is used in databases."}',
 '{"prompt": "Describe REST API", "chosen": "REST API is an architectural style using HTTP methods like GET, POST, PUT, DELETE for communication. This is important for exams and practical understanding.", "rejected": "REST API is used for web services."}',
 '{"prompt": "Define deadlock in OS", "chosen": "Deadlock occurs when processes block each other by holding resources while waiting for others. This is important for exams and practical understanding.", "rejected":

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cuda'

In [7]:
#model_name = "mistralai/Mistral-7B-Instruct-v0.2"
model_name = "meta-llama/Llama-3.2-1B-Instruct"

In [8]:
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
ref_model = AutoModelForCausalLM.from_pretrained(model_name)
ref_model.to(device)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro